# import Libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
%matplotlib inline

c:\Users\86157\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df_weather = pd.read_csv('data/cleaned_weather_data.csv')
df_charging=pd.read_csv('data/cleaned_chargingdata.csv')

In [3]:
print(df_charging.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80639 entries, 0 to 80638
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0.1        80639 non-null  int64  
 1   Unnamed: 0          80639 non-null  int64  
 2   id                  80639 non-null  object 
 3   connectionTime      80639 non-null  object 
 4   disconnectTime      80639 non-null  object 
 5   doneChargingTime    75329 non-null  object 
 6   kWhDelivered        80639 non-null  float64
 7   sessionID           80639 non-null  object 
 8   siteID              80639 non-null  int64  
 9   spaceID             80639 non-null  object 
 10  stationID           80639 non-null  object 
 11  timezone            80639 non-null  object 
 12  userID              63376 non-null  float64
 13  WhPerMile           63376 non-null  float64
 14  kWhRequested        63376 non-null  float64
 15  milesRequested      63376 non-null  float64
 16  minu

In [4]:
print(df_charging.head)

<bound method NDFrame.head of        Unnamed: 0.1  Unnamed: 0                        id  \
0                 0           0  5e23b149f9af8b5fe4b973cf   
1                 1           1  5e23b149f9af8b5fe4b973d0   
2                 2           2  5e23b149f9af8b5fe4b973d1   
3                 3           3  5e23b149f9af8b5fe4b973d2   
4                 4           3  5e23b149f9af8b5fe4b973d2   
...             ...         ...                       ...   
80634         80634       10083  5d574ad2f9af8b4c10c03652   
80635         80635       10084  5d574ad2f9af8b4c10c03653   
80636         80636       10085  5d574ad2f9af8b4c10c03654   
80637         80637       10086  5d574ad2f9af8b4c10c03655   
80638         80638       10087  5d574ad2f9af8b4c10c03656   

                  connectionTime             disconnectTime  \
0      2020-01-02 05:08:54-08:00  2020-01-02 11:11:15-08:00   
1      2020-01-02 05:36:50-08:00  2020-01-02 14:38:21-08:00   
2      2020-01-02 05:56:35-08:00  2020-01-02 16:

In [5]:
print(df_weather.head())

   Unnamed: 0                  timestamp  temperature  cloud_cover  \
0           0  2018-04-25 03:42:00-07:00         12.0         27.0   
1           1  2018-04-25 03:53:00-07:00         12.0         26.0   
2           2  2018-04-25 04:53:00-07:00         12.0         27.0   
3           3  2018-04-25 05:10:00-07:00         12.0         27.0   
4           4  2018-04-25 05:53:00-07:00         12.0         28.0   

  cloud_cover_description  pressure  windspeed  precipitation  \
0           Mostly Cloudy    989.11       11.0            0.0   
1                  Cloudy    989.11       11.0            0.0   
2           Mostly Cloudy    989.11        6.0            0.0   
3           Mostly Cloudy    989.11        7.0            0.0   
4           Mostly Cloudy    989.11        9.0            0.0   

   felt_temperature  
0              12.0  
1              12.0  
2              12.0  
3              12.0  
4              12.0  


In [6]:
print(df_weather.columns)

Index(['Unnamed: 0', 'timestamp', 'temperature', 'cloud_cover',
       'cloud_cover_description', 'pressure', 'windspeed', 'precipitation',
       'felt_temperature'],
      dtype='object')


In [7]:
print(df_weather['timestamp'].dtype)
print(df_charging['connectionTime'].dtype)

object
object


In [8]:
df_charging['connectionTime'] = pd.to_datetime(df_charging['connectionTime'],utc=True)
df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'],utc=True)

In the United States, Daylight Saving Time (DST) transitions cause the clock to skip forward by one hour. To address this, first convert the time columns in both DataFrames to UTC. Perform the time calculations and duplication operations in UTC, and then convert the results back to the Los Angeles time zone.

In [9]:
print(df_weather['timestamp'].dtype)
print(df_charging['connectionTime'].dtype)

datetime64[ns, UTC]
datetime64[ns, UTC]


In [10]:
max_time_weather = df_weather['timestamp'].max()
min_time_weather = df_weather['timestamp'].min()
max_time_charging = df_charging['connectionTime'].max()
min_time_charging = df_charging['connectionTime'].min()
print(max_time_weather,min_time_weather,max_time_charging,min_time_charging)

2021-01-01 07:53:00+00:00 2018-04-25 10:42:00+00:00 2021-09-14 05:43:39+00:00 2018-04-25 11:08:04+00:00


In [11]:
start_time = max_time_weather - pd.DateOffset(years=1)
end_time = max_time_charging - pd.DateOffset(years=1)
rows_to_duplicate = df_weather[(df_weather['timestamp'] >= start_time) &(df_weather['timestamp'] <= end_time)].copy()
rows_to_duplicate['timestamp'] = rows_to_duplicate['timestamp'] + pd.DateOffset(years=1)

In [12]:
df_weather = pd.concat([df_weather, rows_to_duplicate], ignore_index=True)
df_weather = df_weather.reset_index(drop=True)

In [13]:
print(df_weather['timestamp'].dtype)
print(df_charging['connectionTime'].dtype)

datetime64[ns, UTC]
datetime64[ns, UTC]


In [14]:
print(df_charging.describe)

<bound method NDFrame.describe of        Unnamed: 0.1  Unnamed: 0                        id  \
0                 0           0  5e23b149f9af8b5fe4b973cf   
1                 1           1  5e23b149f9af8b5fe4b973d0   
2                 2           2  5e23b149f9af8b5fe4b973d1   
3                 3           3  5e23b149f9af8b5fe4b973d2   
4                 4           3  5e23b149f9af8b5fe4b973d2   
...             ...         ...                       ...   
80634         80634       10083  5d574ad2f9af8b4c10c03652   
80635         80635       10084  5d574ad2f9af8b4c10c03653   
80636         80636       10085  5d574ad2f9af8b4c10c03654   
80637         80637       10086  5d574ad2f9af8b4c10c03655   
80638         80638       10087  5d574ad2f9af8b4c10c03656   

                 connectionTime             disconnectTime  \
0     2020-01-02 13:08:54+00:00  2020-01-02 11:11:15-08:00   
1     2020-01-02 13:36:50+00:00  2020-01-02 14:38:21-08:00   
2     2020-01-02 13:56:35+00:00  2020-01-02 16:

In [15]:
def find_closest_weather(row):
    connection_time = row['connectionTime']
    time_diff = (df_weather['timestamp'] - connection_time).abs()
    closest_index = time_diff.idxmin()
    return df_weather.loc[closest_index]
closest_weather = df_charging.apply(find_closest_weather, axis=1)
weather_columns = ['timestamp', 'temperature', 'cloud_cover', 'cloud_cover_description', 'pressure', 'windspeed', 'precipitation','felt_temperature']
for col in weather_columns:
    df_charging[col] = closest_weather[col].values

This code aims to find the closest weather record (df_weather) for each charging record (df_charging) based on its charging time (connectionTime). The implementation steps include:

Define the function find_closest_weather:
Takes a row from the charging record and calculates the time difference between connectionTime and weather timestamps.
Returns the weather record with the smallest time difference.

Apply the function using apply:
Applies the find_closest_weather function to each row in df_charging, producing the closest weather record for each charging record.

Add weather data to charging records:
Appends weather data (e.g., timestamp, temperature, cloud cover) column by column to df_charging.

In [16]:
print(df_weather['timestamp'].dtype)
print(df_charging['connectionTime'].dtype)

datetime64[ns, UTC]
datetime64[ns, UTC]


In [17]:
print(df_charging.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80639 entries, 0 to 80638
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   Unnamed: 0.1             80639 non-null  int64              
 1   Unnamed: 0               80639 non-null  int64              
 2   id                       80639 non-null  object             
 3   connectionTime           80639 non-null  datetime64[ns, UTC]
 4   disconnectTime           80639 non-null  object             
 5   doneChargingTime         75329 non-null  object             
 6   kWhDelivered             80639 non-null  float64            
 7   sessionID                80639 non-null  object             
 8   siteID                   80639 non-null  int64              
 9   spaceID                  80639 non-null  object             
 10  stationID                80639 non-null  object             
 11  timezone                 806

In [18]:
max_row = df_charging[df_charging['connectionTime'] == max_time_charging]
print(max_row)

       Unnamed: 0.1  Unnamed: 0                        id  \
25630         25630        5875  6155053bf9af8b76960e16d1   

                 connectionTime             disconnectTime  \
25630 2021-09-14 05:43:39+00:00  2021-09-14 07:46:28-07:00   

                doneChargingTime  kWhDelivered  \
25630  2021-09-14 07:46:22-07:00        53.937   

                                    sessionID  siteID  spaceID  ...  \
25630  1_1_178_817_2021-09-14 05:43:27.354300       1  AG-1F09  ...   

      paymentRequired         requestedDeparture           timestamp  \
25630            True  2021-09-14 10:15:39-07:00 2021-09-14 04:53:00   

       temperature  cloud_cover  cloud_cover_description  pressure windspeed  \
25630         22.0         33.0                     Fair    985.16       9.0   

       precipitation felt_temperature  
25630            0.0             22.0  

[1 rows x 28 columns]


In [19]:
df_charging['connectionTime']=df_charging['connectionTime'].dt.tz_convert('America/Los_Angeles')
df_weather['timestamp']=df_weather['timestamp'].dt.tz_convert('America/Los_Angeles')
print(df_weather['timestamp'].dtype)
print(df_charging['connectionTime'].dtype)

datetime64[ns, America/Los_Angeles]
datetime64[ns, America/Los_Angeles]


In [20]:
missing_count = df_charging['connectionTime'].isna().sum()
print(missing_count)

0


In [21]:
df_charging.to_csv('merged_data.csv')